# 09 - SW4 export: single point source + real stations

This notebook builds the **simplest** ShakerMaker-to-SW4 case:

- a single **PointSource** at the `Centro` location (depth 10 km),
- a set of receiver **Stations** at the other real UTM coordinates,
- a 4-layer crust,

then exports it to SW4 with `model.export_sw4(...)` and plots the geometry in
the local km frame. Every figure is saved as a `.png` in this folder.

The SW4 export only **writes input files** (it does not run the FK core), so
it is fast.

## Coordinate convention

ShakerMaker axes are `x = North`, `y = East`, `z = Down`. The station data is
in UTM, where `x = East`, `y = North`. We center a **local km frame** on
`Centro` (index 0):

```
x_km(North) = (utmy - utmy[0]) / 1000.0
y_km(East)  = (utmx - utmx[0]) / 1000.0
```

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

from shakermaker.shakermaker import ShakerMaker
from shakermaker.crustmodel import CrustModel
from shakermaker.pointsource import PointSource
from shakermaker.faultsource import FaultSource
from shakermaker.station import Station
from shakermaker.stationlist import StationList
from shakermaker.stf_extensions.gaussian import Gaussian

## Station data and the UTM -> local km conversion

Index 0 (`Centro`) is the source; the rest are receivers.

In [ ]:
utmx = [359958.1764612976, 359909.210734884, 352972.9064965788, 356785.00778720574, 357388.4436483849, 343765.088895043, 349518.5389304694, 346324.7094952749, 333266.0855400809, 337477.02458516695, 336224.1507329495, 358265.57164]
utmy = [6294124.525366314, 6302625.576311215, 6302517.54834607, 6293263.310727202, 6283866.121857278, 6306996.823725268, 6293815.479730421, 6282778.22420124, 6304244.590630547, 6292761.800243624, 6278203.501919601, 6300158.0242]
names = ['Centro', 'H1_s0', 'N1_s1', 'N2_s2', 'N3_s3', 'I1_s4', 'I2_s5', 'I3_s6', 'F1_s7', 'F2_s8', 'F3_s9', 'CEN']

x_km = [(utmy[i] - utmy[0]) / 1000.0 for i in range(len(utmy))]  # North
y_km = [(utmx[i] - utmx[0]) / 1000.0 for i in range(len(utmx))]  # East

for i in range(len(names)):
    print(f"{names[i]:8s}  x(N)={x_km[i]:8.3f} km  y(E)={y_km[i]:8.3f} km")

## Crust, source and stations

A 4-layer crust (thin slow layers over a faster half-space). A single
`PointSource` at `Centro` (local 0, 0) with depth `z = 10 km` and a Gaussian
source-time function. The other UTM coordinates become receiver stations.

The STF needs a sampling step `dt` before the export can discretise it.

In [ ]:
crust = CrustModel(4)
crust.add_layer(0.200, 1.32, 0.75, 2.40, 1000.0, 1000.0)
crust.add_layer(0.800, 2.75, 1.57, 2.50, 1000.0, 1000.0)
crust.add_layer(14.500, 5.50, 3.14, 2.50, 1000.0, 1000.0)
crust.add_layer(0.000, 7.00, 4.00, 2.67, 1000.0, 1000.0)
print(crust)

stf = Gaussian(t0=0.36, freq=16.6667, M0=1.0)
stf.dt = 0.01
src = PointSource([x_km[0], y_km[0], 10.0], [0.0, 90.0, 0.0], stf=stf)
fault = FaultSource([src], metadata={"name": "centro_source"})

stations = [Station([x_km[i], y_km[i], 0.0], metadata={"name": names[i]}) for i in range(1, len(names))]
stationlist = StationList(stations, metadata={"name": "stg_stations"})

model = ShakerMaker(crust, fault, stationlist)

## Preview the model before exporting
We inspect the crust (layered column + velocity profile), the source, and its source time function (STF) before exporting to SW4. Each figure is saved as a PNG here.

In [ ]:
import matplotlib.pyplot as plt
from shakermaker.tools.plotting import SourcePlot

dt = 0.01  # STF sampling step used by the SW4 export

crust.plot()
plt.gcf().savefig("crust_layers.png", dpi=150, bbox_inches="tight")

crust.plot_profile()
plt.gcf().savefig("crust_velocity_profile.png", dpi=150, bbox_inches="tight")

for _s in fault:
    _s.stf.dt = dt
SourcePlot(fault, show=False).savefig("source_geometry.png", dpi=150, bbox_inches="tight")

_stf = fault.get_source_by_id(0).stf
fig_stf, ax_stf = plt.subplots(figsize=(6, 2.5))
ax_stf.plot(_stf.t, _stf.data, color="tab:red", lw=1.5)
ax_stf.set_xlabel("time (s)")
ax_stf.set_ylabel("amplitude")
ax_stf.set_title("Source time function (STF)")
fig_stf.tight_layout()
fig_stf.savefig("source_stf.png", dpi=150, bbox_inches="tight")

## Geometry in the local km frame (map view)

Source as a red star, receivers as blue triangles, labeled by name.
East on the horizontal axis, North on the vertical axis.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(y_km[0], x_km[0], marker='*', s=300, c='red', zorder=3, label='source (Centro, z=10 km)')
ax.scatter(y_km[1:], x_km[1:], marker='^', s=80, c='tab:blue', zorder=2, label='receivers')
for i in range(len(names)):
    ax.annotate(names[i], (y_km[i], x_km[i]), textcoords='offset points', xytext=(5, 5), fontsize=8)
ax.set_xlabel('y = East (km)')
ax.set_ylabel('x = North (km)')
ax.set_title('SW4 export geometry (local km frame, centered on Centro)')
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.legend()
plt.gcf().savefig('sw4_geometry_map.png', dpi=150, bbox_inches='tight')

## Geometry in 3D (depth down)

The source sits at 10 km depth below `Centro`; the receivers sit at the
surface (z = 0). The vertical axis is inverted so depth grows downward.

In [ ]:
fig = plt.figure(figsize=(8, 7))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(y_km[0], x_km[0], 10.0, marker='*', s=300, c='red', label='source (z=10 km)')
ax.scatter(y_km[1:], x_km[1:], np.zeros(len(names) - 1), marker='^', s=60, c='tab:blue', label='receivers')
ax.set_xlabel('y = East (km)')
ax.set_ylabel('x = North (km)')
ax.set_zlabel('z = Depth (km)')
ax.invert_zaxis()
ax.set_title('SW4 export geometry (3D, z down)')
ax.legend()
plt.gcf().savefig('sw4_geometry_3d.png', dpi=150, bbox_inches='tight')

## Export to SW4

`export_sw4` writes the SW4 input file, per-source slip-rate files and a
compact HDF5 transport package under `<path>/shakermakerexports/`. The grid
spacing `h` is in **meters**; `size_domain = [x_North, y_East, z_Down]` is in
meters too. Nothing here runs the FK core.

In [ ]:
out_dir = os.path.join(os.getcwd(), '_sw4_out_nb')

model.export_sw4(
    path=out_dir,
    h=100.0,
    size_domain=[40000.0, 40000.0, 25000.0],
    tmax=40.0,
    m0=1.0,
    station_prefix='sf',
    shakermaker_stations=True,
    h5_export_name='sw4_package.h5',
    plot_geometry=False,
    plot_geometry_sw4=False,
)

package = os.path.join(out_dir, 'shakermakerexports', 'sw4_package.h5')
assert os.path.exists(package), f'missing package: {package}'
print('package:', package)

## Inspect the package

Peek at the SW4 input text and the receiver coordinates stored in the
transport package (requires `h5py`).

In [ ]:
try:
    import h5py
    with h5py.File(package, 'r') as f:
        text = f['sw4_input/text'][()]
        text = text.decode('utf-8') if isinstance(text, bytes) else str(text)
        print(text[:600])
        print('...')
        print('receivers:', len(f['receivers/id'][:]))
except Exception as exc:
    print('h5py not available or read failed:', exc)

## Optional: SW4 geometry plot via the exporter

The exporter can render its own geometry view (it uses pyvista when
available). Re-run the export with `plot_geometry=True` only if pyvista is
installed; otherwise skip silently.

In [ ]:
try:
    import pyvista  # noqa: F401
    model.export_sw4(
        path=out_dir,
        h=100.0,
        size_domain=[40000.0, 40000.0, 25000.0],
        tmax=40.0,
        m0=1.0,
        shakermaker_stations=True,
        plot_geometry=True,
    )
except Exception as exc:
    print('SKIP exporter geometry plot:', exc)